# HumorVibes — Mesh Zoo Lab (keyless multi-family panel)

Four model families as **local Kaggle weights** — no API keys: Gemma-2-2B (reference instrument), Gemma-3-1B, Llama-3.2-3B, Qwen2.5-1.5B.

1. **Frame duel v2**: every family writes frames; the instrument scores them in nats (null-controlled).
2. **Cross-instrument invariance**: S/R re-measured off Llama's logits — is the laugh region a property of jokes, or of one model?
3. **Century test**: 1916 public-domain jokes measured today (THEORY.md §11).

In [ ]:
import gc, glob, json, os, re, time, urllib.request, torch
from transformers import AutoModelForCausalLM, AutoTokenizer
os.makedirs('/kaggle/working/research_out', exist_ok=True)

def find_model(*name_parts):
    for cfg in sorted(glob.glob('/kaggle/input/**/config.json', recursive=True)):
        low = cfg.lower()
        if all(p in low for p in name_parts):
            return os.path.dirname(cfg)
    return None

ZOO = {k: v for k, v in {
    'gemma-2-2b': find_model('gemma-2', '2b'),
    'gemma-3-1b': find_model('gemma-3', '1b'),
    'llama-3.2-3b': find_model('llama', '3b'),
    'qwen2.5-1.5b': find_model('qwen', '1.5b'),
}.items() if v}
print('zoo mounts:', json.dumps(ZOO, indent=1))

CUR = {'name': None, 'tok': None, 'model': None}
def load(name):
    if CUR['name'] == name: return
    if CUR['model'] is not None:
        del CUR['model']; CUR['model'] = None; gc.collect()
        torch.cuda.empty_cache() if torch.cuda.is_available() else None
    path = ZOO[name]
    tok = AutoTokenizer.from_pretrained(path)
    model = None
    if torch.cuda.is_available():
        try:
            model = AutoModelForCausalLM.from_pretrained(path, torch_dtype=torch.float16, device_map='auto').eval()
            with torch.no_grad(): model(torch.tensor([[tok.eos_token_id or 2]]).to(model.device))
        except Exception as e:
            print(name, 'cuda->cpu:', str(e)[:80]); model = None; torch.cuda.empty_cache()
    if model is None:
        model = AutoModelForCausalLM.from_pretrained(path, torch_dtype=torch.float32).eval()
    CUR.update(name=name, tok=tok, model=model)
    print(f'loaded {name} on {model.device}')

def nll_mean(context, continuation):
    tok, model = CUR['tok'], CUR['model']
    ctx = tok(context, return_tensors='pt').input_ids
    cont = tok(continuation, add_special_tokens=False, return_tensors='pt').input_ids
    full = torch.cat([ctx, cont], dim=1).to(model.device)
    with torch.no_grad():
        lp = torch.log_softmax(model(full).logits.float(), dim=-1)
    n = ctx.shape[1]
    vals = [float(-lp[0, n+i-1, int(full[0, n+i])]) for i in range(cont.shape[1])]
    return sum(vals) / len(vals)

def gen(prompt, max_new=60, temperature=0.4):
    tok, model = CUR['tok'], CUR['model']
    try:
        ids = tok.apply_chat_template([{'role':'user','content':prompt}], return_tensors='pt',
                                      add_generation_prompt=True)
        if not torch.is_tensor(ids): ids = ids['input_ids']
    except Exception:
        ids = tok(prompt, return_tensors='pt').input_ids
    ids = ids.to(model.device)
    with torch.no_grad():
        out = model.generate(ids, max_new_tokens=max_new, do_sample=True, temperature=temperature,
                             top_p=0.95, pad_token_id=tok.eos_token_id or tok.pad_token_id)
    return tok.decode(out[0, ids.shape[1]:], skip_special_tokens=True).strip()

DECOY = 'It turns out this is really about quarterly regional cheese sales figures.'
def measure_frame(setup, punch, frame):
    S = nll_mean(setup + '\n', ' ' + punch)
    r_raw = max(0.0, S - nll_mean(setup + '\n(' + frame + ')\n', ' ' + punch))
    # leaky-frame guard: only count hint overlap against the punchline's
    # NOVEL words (present in punch but NOT in setup - setup words are
    # fair reuse); v1 finding: qwen confabulation scored 2.23 net-of-decoy
    # on nonsense via lexical overlap
    pw = {w.lower().strip('.,!?\"\'' ) for w in punch.split() if len(w) > 3}
    sw = {w.lower().strip('.,!?\"\'' ) for w in setup.split() if len(w) > 3}
    pw_novel = pw - sw
    fw = {w.lower().strip('.,!?\"\'' ) for w in frame.split() if len(w) > 3}
    leak = len(pw_novel & fw) / max(1, len(pw_novel))
    if leak > 0.4: r_raw *= max(0.0, 1.0 - leak)
    r_null = max(0.0, S - nll_mean(setup + '\n(' + DECOY + ')\n', ' ' + punch))
    return round(S, 3), round(max(0.0, r_raw - r_null), 3)

In [ ]:
ITEMS = [
  ('speed_bumps', 'I told my therapist about my fear of speed bumps.', "She said I'm slowly getting over it.",
   "'Getting over it' is literal: the car physically drives over the speed bumps slowly."),
  ('lion_heart', 'My grandfather has the heart of a lion', 'and a lifetime ban from the zoo.',
   "He literally stole a lion's heart from the zoo, not the bravery metaphor."),
  ('ai_pm', 'I asked the AI project manager when the feature would ship.', 'It scheduled a meeting to align on what \'when\' means.',
   'The AI treats even the word when as a project requirement needing stakeholder alignment.'),
  ('nonsense_ctrl', 'I told my therapist about my fear of speed bumps.', 'The quarterly report shows strong regional cheese sales.', 'NONE'),
]
FRAME_ASK = ('A joke works because a hidden frame reinterprets the punchline - the fact that, once stated, '
             'makes the punchline the OBVIOUS next thing to say. Do NOT restate the punchline; name the '
             'reinterpretation. If the two parts are simply unrelated, the honest answer is NONE.\n'
             "Example - Joke: I told my therapist about my fear of speed bumps. She said I'm slowly getting over it. "
             "Frame: 'Getting over it' is literal - the car drives over the bumps slowly.\n"
             'Example - Joke: I bought new shoes. The harvest in Portugal was strong this year. Frame: NONE\n'
             'Joke: {joke}\nFrame (ONE short sentence; if none exists, write NONE):')

# Stage A: every zoo family writes frames (collect text only, one model in memory at a time)
frames = {}
for name in ZOO:
    load(name)
    frames[name] = {}
    for iid, s, p, _gt in ITEMS:
        out = gen(FRAME_ASK.format(joke=s + ' ' + p), max_new=50, temperature=0.3)
        frames[name][iid] = (out.splitlines()[0].strip() if out else 'NONE')[:120]
    print(name, 'frames done')

## 1. Frame duel v2 — scored by the Gemma-2 reference instrument

In [ ]:
load('gemma-2-2b')
duel = {}
for iid, s, p, gt in ITEMS:
    rows = {}
    if gt != 'NONE':
        S, R = measure_frame(s, p, gt)
        rows['ground_truth'] = {'frame': gt, 'S': S, 'R': R}
    for writer in frames:
        f = frames[writer][iid]
        if not f or f.upper().startswith('NONE'):
            rows[writer] = {'frame': 'NONE', 'R': 0.0,
                            'honest_none': iid == 'nonsense_ctrl'}
            continue
        S, R = measure_frame(s, p, f)
        rows[writer] = {'frame': f, 'S': S, 'R': R}
    duel[iid] = rows
    print('==', iid)
    for w, r in sorted(rows.items(), key=lambda kv: -(kv[1].get('R') or 0)):
        print(f"   {w:16s} R={r.get('R', 0):5.2f} :: {str(r['frame'])[:64]}")
deficits = {}
for w in frames:
    ds = [max(0.0, duel[i]['ground_truth']['R'] - (duel[i].get(w, {}).get('R') or 0.0))
          for i, _, _, gt in [(x[0], x[1], x[2], x[3]) for x in ITEMS] if gt != 'NONE']
    deficits[w] = round(sum(ds) / len(ds), 3)
print('\nEXPLANATION DEFICIT (lower = better writer):', json.dumps(deficits, indent=1))

## 2. Cross-instrument invariance — Llama measures what Gemma measured

In [ ]:
alt = 'llama-3.2-3b' if 'llama-3.2-3b' in ZOO else ('qwen2.5-1.5b' if 'qwen2.5-1.5b' in ZOO else None)
invariance = {}
if alt:
    gem = {iid: (duel[iid]['ground_truth']['S'], duel[iid]['ground_truth']['R'])
           for iid, _, _, gt in ITEMS if gt != 'NONE'}
    load(alt)
    altm = {}
    for iid, s, p, gt in ITEMS:
        if gt == 'NONE': continue
        altm[iid] = measure_frame(s, p, gt)
    print(f'{alt} as instrument:')
    for iid in gem:
        print(f'  {iid:14s} gemma S={gem[iid][0]:5.2f} R={gem[iid][1]:5.2f} | {alt} S={altm[iid][0]:5.2f} R={altm[iid][1]:5.2f}')
    # rank agreement on R (n=3: report orderings, not pretend-statistics)
    gr = sorted(gem, key=lambda i: -gem[i][1]); ar = sorted(altm, key=lambda i: -altm[i][1])
    invariance = {'gemma_R_order': gr, f'{alt}_R_order': ar, 'same_order': gr == ar,
                  'gemma': {k: gem[k] for k in gem}, alt: {k: altm[k] for k in altm}}
    print('R-ordering identical across instruments:', gr == ar)

## 3. Century test — 1916 jest-book items measured today (public domain)

In [ ]:
century = []
try:
    req = urllib.request.Request('https://www.gutenberg.org/cache/epub/18464/pg18464.txt',
                                 headers={'User-Agent': 'HumorVibes research (Kaggle notebook)'})
    text = urllib.request.urlopen(req, timeout=30).read().decode('utf-8', 'replace')
    body = re.split(r'\*\*\* ?START[^\n]*\n', text)[-1]
    body = re.split(r'\*\*\* ?END', body)[0]
    blocks = [' '.join(b.split()) for b in re.split(r'\n\s*\n', body)]
    blocks = blocks[len(blocks) // 4:]  # drop first 25%: front matter/preface, not jokes
    def _pref(b):
        return '?' in b or any(w in b.lower() for w in ('said', 'replied', 'asked'))
    pool = [b for b in blocks if 80 <= len(b) <= 320 and '"' in b and b[-1] in '.!?"\'' and not b.isupper()]
    cands = sorted(pool, key=lambda b: not _pref(b))[:12]
    def split_sp(t):
        for sep in ['. ', '? ', '! ', ': ']:
            if sep in t:
                a, b = t.rsplit(sep, 1)
                if len(b.split()) >= 2: return a + sep.strip(), b.strip()
        w = t.split(); c = max(1, int(len(w)*0.7)); return ' '.join(w[:c]), ' '.join(w[c:])
    best_writer = min(deficits, key=deficits.get)
    print('century writer (lowest deficit):', best_writer, deficits.get(best_writer))
    load(best_writer)
    written = []
    for b in cands:
        s, p = split_sp(b)
        frame = gen(FRAME_ASK.format(joke=b), max_new=50, temperature=0.3).splitlines()[0].strip()
        written.append((b, s, p, frame))
    load('gemma-2-2b')
    for b, s, p, frame in written:
        if not frame or frame.upper().startswith('NONE'):
            century.append({'joke': b[:100], 'frame': 'NONE', 'R': 0.0, 'writer': best_writer}); continue
        S, R = measure_frame(s, p, frame)
        century.append({'joke': b[:100], 'frame': frame[:80], 'S': S, 'R': R, 'writer': best_writer})
    century.sort(key=lambda r: -(r.get('R') or 0))
    alive = [c for c in century if (c.get('R') or 0) >= 0.3]
    print(f'1916 corpus: {len(century)} items (frames by {best_writer}) | alive today: {len(alive)}')
    for c in century[:6]:
        print(f"  R={c.get('R',0):5.2f} :: {c['joke'][:78]}")
except Exception as e:
    print('century test skipped:', e)
json.dump({'duel': duel, 'deficits': deficits, 'invariance': invariance, 'century': century},
          open('/kaggle/working/research_out/zoo_report.json', 'w'), indent=2)
print('wrote zoo_report.json')

## Reading the results
- **Deficit leaderboard**: which family explains jokes best, in nats — with zero API keys.
- **Invariance**: if S/R orderings agree across instruments, the laugh region is a property of the jokes, not an artifact of Gemma's logits — the theory's strongest robustness card.
- **Century test**: frames renting universal mechanics survive 110 years; frames renting dead caches don't (THEORY.md §11's first falsifiable temporal prediction).